In [2]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rc('xtick', direction='in', top=True)
mpl.rc('ytick', direction='in', right=True)
mpl.rc('xtick.minor', visible=True)
mpl.rc('ytick.minor', visible=True)
import os
import json
rng = np.random.default_rng()



# input and output Creation Func

In [3]:
def more_feature_inputs_outputs(index1,index2,spread=9,noise=False,SNR=1,verbose=False,minpeaks=True,freq_diff_features=False):
    """
    This Function takes 2 premade Jason indexes containing a file structure of the angle position and then histograms of a simulation data ROOT file
    The method to create these jason files can be found in data_load.ipynb within this same directory.

    Inputs

    Index1: the TE011 Jason index
    Index2: The TE012 Jason index

    spread=9 : this is used in several points it gneraly refers to the number of frequency bins arround a central point that can be considerd within the same peak or is likly to be that same peak
    Noise=False: When true adds noise to the PSDs based on an SNR from the highest peak
    SNR=1: said SNR
    verbose=False: if true prints an update on progress as the fuction runs
    minpeaks=True: When true will output 5 relvent peaks corisponding to the 5 posible sidebands when false will output 10 relivent peaks 5 from TE011 5 from TE012
    Freq_diff_features=False: adds features to the output of every frequenceas difference to every other frequency

    Outputs

    all_sidebands: a nx5x2 array of the location and power of each sideband in the order -2,0,2,-1,1

    Inputs_sorted: an array of the classifier inputs sorted by frequency npsds x nPeaks x nfeatures

    Outputs_sorted: an array of the classifier outputs npsd x nPeaks with labels -2,-1,0,1,2,np.inf corisponding to the input peak at the same locatin if it is a sideband with said lable or junk/noise np.inf


    """

    #Spread needs to be odd
    if spread % 2 != 1:
        spread += 1

    # arrays of the "locations" of sidebands
    band1 = np.array([-2, 0, 2,])
    band2 = np.array([-1,1])

    #preparing arrays
    all_sidebands = []
    all_maxbands = []
    all_outputs=[]


    jsn_name = "LocustEventProperties.json"

    #Begin iteration through angles the angles for Index1 and Index2 should be the same
    for angle in index1:

        #Begin iteration through position the angles for Index1 and Index2 should be the same
        for position in index1[angle]:

            #Progress printing
            if verbose:
                print(f"Sidebands:Processing angle={angle}, position={position}", flush=True)

            #loading relevent information from the event properties jason file
            jsn_info = index1[angle][position][jsn_name]
            jsn_filepath = jsn_info["filepath"]

            with open(jsn_filepath, "r") as f:
                jsn = json.load(f)
        
            avgax = jsn["0"]["0"]["avg-axial-frequency"]
            central = jsn["0"]["0"]["output-avg-frequency"]

            #Loading and storing all of the filepaths for each PSD
            psd_by_filepath1 = {}
            psd_by_filepath2 = {}

            for hist_name, hist_info in index1[angle][position].items():
                if "PSD" not in hist_name:
                    continue

                filepath = hist_info["filepath"]

                if filepath not in psd_by_filepath1:
                    psd_by_filepath1[filepath] = []

                psd_by_filepath1[filepath].append(hist_name)  
            
            for hist_name, hist_info in index2[angle][position].items():
                if "PSD" not in hist_name:
                    continue

                filepath = hist_info["filepath"]

                if filepath not in psd_by_filepath2:
                    psd_by_filepath2[filepath] = []

                psd_by_filepath2[filepath].append(hist_name)

            # Begin Itteration through each file

            for (filepath1, hist_names), (filepath2, _) in zip(psd_by_filepath1.items(), psd_by_filepath2.items()):
                file1 = ROOT.TFile.Open(filepath1)
                file2 = ROOT.TFile.Open(filepath2)
                if not file1 or file1.IsZombie():
                    raise RuntimeError(f"Could not open ROOT file: {filepath}")
                if not file2 or file2.IsZombie():
                    raise RuntimeError(f"Could not open ROOT file: {filepath}")
                try:
                    # Begin Itteration through each Hist

                    for hist_name in hist_names:
                        hist1 = file1.Get(hist_name)
                        hist2 = file2.Get(hist_name)
                        if not hist1 or not hist2:
                            raise RuntimeError(
                                f"Missing histogram {hist_name} in file {filepath}"
                            )
                        nbins = hist1.GetNbinsX()
                        
                        bins = range(1, nbins + 1)

                        #Loading powers and frequencies from both Hists
                        power1 = []
                        power2 = []
                        center1 = []
                        center2 = []

                        for i in bins:
                            power1.append(hist1.GetBinContent(i))
                            power2.append(hist2.GetBinContent(i))
                            center = hist1.GetBinCenter(i)
                            center1.append(center)
                            center2.append(center)

                        power1 = np.array(power1)
                        power2 = np.array(power2)
                        center1 = np.array(center1)
                        center2 = np.array(center2)

                        space1=np.mean(np.diff(center1))
                        space2=np.mean(np.diff(center2))
                        
                        # Finding and storing the Location and Power of each sideband -2,0,2 from TE011 and -1,1 from TE012

                        sidebands=[]

                        for band in band1:

                            target = central + (avgax * band)
                            mask = np.abs(center1 - target) <= (space1 * spread)

                            if np.any(mask):

                                local_idx = np.argmax(power1[mask])
                                max_idx = np.where(mask)[0][local_idx]
                                power = power1[max_idx]

                                if power == 0:
                                    loc = np.inf
                                    pos=0
                                else:
                                    loc = int(band)
                                    pos=center1[max_idx]

                                result = (power,pos, loc)
                            else:
                                result = (0, 0, np.inf)
                                
                            sidebands.append(result)

                        for band in band2:

                            target = central + (avgax * band)
                            mask = np.abs(center2 - target) <= (space2 * spread)

                            if np.any(mask):
 
                                local_idx = np.argmax(power2[mask])
                                max_idx = np.where(mask)[0][local_idx]
                                power = power2[max_idx]

                                if power == 0:
                                    loc = np.inf
                                    pos=0
                                else:
                                    loc = int(band)
                                    pos=center2[max_idx]

                                result = (power,pos, loc)
                            else:
                                result = (0, 0, np.inf)

                            sidebands.append(result)
                            
                            
                        sidebands = np.array(sidebands)


                        #Finding and loading all relevent maximum peaks for inputs

                        maxbands1=[]
                        maxbands2=[]
                        power_sim1 = power1.copy()
                        power_sim2 = power2.copy()

                        #Adding Noise  
                        if noise:
                            mx1=np.max(power_sim1)
                            mx2=np.max(power_sim2)
                            sigma1=np.sqrt(mx1/(2*SNR))
                            sigma2=np.sqrt(mx2/(2*SNR))
                            noise_power1 = np.abs(np.random.normal(0, sigma1, nbins) + 1j*np.random.normal(0, sigma1, nbins))**2
                            noise_power2 = np.abs(np.random.normal(0, sigma2, nbins) + 1j*np.random.normal(0, sigma2, nbins))**2
                            
                            power_sim1+=noise_power1
                            power_sim2+=noise_power2

                        max_peaks_found1=0 
                        max_peaks_found2=0 

                        if minpeaks:
                            maxpeaks1=3
                            maxpeaks2=2
                        else:
                            maxpeaks1=5
                            maxpeaks2=5

                        while max_peaks_found1<maxpeaks1:

                            mx=np.argmax(power_sim1)
                            center=float(center1[mx]) 
                            power=float(power_sim1[mx])

                            if len(maxbands1) > 0:
                                matches = np.where(
                                    np.abs(np.array(maxbands1)[:,1] - center) <= space1 * spread)[0]
                            else:
                                matches = []

                            if len(matches) == 1 and power !=0 :
                                maxbands1[matches[0]][0]+=power
                            else:
                                maxbands1.append([power,center])
                                max_peaks_found1+=1
                            
                            power_sim1[mx] = 0
                            
                        while max_peaks_found2<maxpeaks2:

                            mx=np.argmax(power_sim2)
                            center=float(center2[mx]) 
                            power=float(power_sim2[mx])

                            if len(maxbands2) > 0:
                                matches = np.where(
                                    np.abs(np.array(maxbands2)[:,1] - center) <= space2 * spread)[0]
                            else:
                                matches = []

                            if len(matches) == 1 and power !=0 :
                                maxbands2[matches[0]][0]+=power
                            else:
                                maxbands2.append([power,center])
                                max_peaks_found2+=1

                            power_sim2[mx] = 0

                        maxbands1=np.array(maxbands1)
                        maxbands2=np.array(maxbands2)
                        
                        maxbands1_sorted = maxbands1[np.argsort(maxbands1[:, 1])]
                        maxbands2_sorted = maxbands2[np.argsort(maxbands2[:, 1])]

                        maxbands = np.vstack((maxbands1_sorted, maxbands2_sorted))

                        # Creating outputs by coparing sidebands and maxbands to find matching peaks

                        ar1=maxbands
                        ar2=sidebands
                        match_matrix = np.isclose(ar1[:, None, 1],ar2[None, :, 1],rtol=0,atol=1e-6)
                        ar1_i, ar2_i = np.where(match_matrix)
                        out = np.full(ar1.shape[0], np.inf)
                        out[ar1_i] = ar2[ar2_i, 2]
                        if minpeaks:
                        
                            out[:3][np.isin(out[:3], [-1, 1])] = np.inf
                            out[3:][np.isin(out[3:], [-2, 0, 2])] = np.inf
                        else:
                
                            out[:5][np.isin(out[:5], [-1, 1])] = np.inf
                            out[5:][np.isin(out[5:], [-2, 0, 2])] = np.inf
                        
                        sort_idx = np.argsort(maxbands[:, 1])
                        maxbands_sorted = maxbands[sort_idx]
                        out_out=out[sort_idx]
                        sidebands_out=sidebands[np.argsort(sidebands[:,1])]

                        # Adding Frequnecy Difference features if requensted
                        if freq_diff_features:

                            freq = maxbands_sorted[:, 1]
                            diffs = freq[:, None] - freq[None, :]  

                            maxbands_sorted = np.hstack((maxbands_sorted, diffs))
                            maxbands_out=maxbands_sorted

                        else:
                            maxbands_out=maxbands_sorted


                        

                        all_sidebands.append(sidebands_out)
                        all_maxbands.append(maxbands_out)
                        all_outputs.append(out_out)
                finally:
                    file1.Close()
                    file2.Close()
    all_sidebands=np.array(all_sidebands)
    all_maxbands=np.array(all_maxbands)
    all_outputs=np.array(all_outputs)

    return(all_sidebands,all_maxbands,all_outputs)

# Input and Output Creation

In [4]:
base_path1="Band_class_training data/TE_011/Results"
index_path1 = os.path.join(base_path1, "Te011.json")
with open(index_path1, "r") as f:
    TE011_Index = json.load(f)

base_path2 = "Band_class_training data/TE_012/Results"
index_path2 = os.path.join(base_path2, "Te012.json")
with open(index_path2, "r") as f:
    TE012_Index = json.load(f)

In [5]:
sidebands1,model_inputs1,model_outputs1=more_feature_inputs_outputs(TE011_Index,TE012_Index,freq_diff_features=True,verbose=True,noise=True,SNR=10)

Sidebands:Processing angle=Angle88.0000000, position=Pos0.0020000
Sidebands:Processing angle=Angle88.0000000, position=Pos0.0040000
Sidebands:Processing angle=Angle88.0000000, position=Pos0.0010000
Sidebands:Processing angle=Angle88.0000000, position=Pos0.0030000
Sidebands:Processing angle=Angle88.0000000, position=Pos0.0050000
Sidebands:Processing angle=Angle88.0000000, position=Pos0.0000000
Sidebands:Processing angle=Angle88.0000000, position=Pos0.0060000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0040000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0050000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0060000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0020000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0000000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0030000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0010000
Sidebands:Processing angle=Angle88.0500000, position=Pos0.0030000
Sidebands:

In [6]:
sidebands2,model_inputs2,model_outputs2=more_feature_inputs_outputs(TE011_Index,TE012_Index,freq_diff_features=True,verbose=True,noise=True,SNR=50)

Sidebands:Processing angle=Angle88.0000000, position=Pos0.0020000
Sidebands:Processing angle=Angle88.0000000, position=Pos0.0040000
Sidebands:Processing angle=Angle88.0000000, position=Pos0.0010000
Sidebands:Processing angle=Angle88.0000000, position=Pos0.0030000
Sidebands:Processing angle=Angle88.0000000, position=Pos0.0050000
Sidebands:Processing angle=Angle88.0000000, position=Pos0.0000000
Sidebands:Processing angle=Angle88.0000000, position=Pos0.0060000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0040000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0050000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0060000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0020000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0000000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0030000
Sidebands:Processing angle=Angle88.0250000, position=Pos0.0010000
Sidebands:Processing angle=Angle88.0500000, position=Pos0.0030000
Sidebands:

In [7]:
sidebands=np.vstack((sidebands1,sidebands2))
model_inputs=np.vstack((model_inputs1,model_inputs2))
model_outputs=np.vstack((model_outputs1,model_outputs2))

# Py torch prep

In [8]:
import torch
import sklearn
import tqdm
import tensorboard

In [9]:
label_to_class = {
    -2: 0,
    -1: 1,
     0: 2,
     1: 3,
     2: 4,
}

class_to_label = {
    0: -2,
    1: -1,
    2: 0,
    3: 1,
    4: 2,
    5: np.inf,
}

def band_encoding(outputs):
    outputs_incoded =np.empty(outputs.shape,dtype=np.int64)

    outputs_incoded[outputs == -2] = 0
    outputs_incoded[outputs == -1] = 1
    outputs_incoded[outputs ==  0] = 2
    outputs_incoded[outputs ==  1] = 3
    outputs_incoded[outputs ==  2] = 4
    outputs_incoded[np.isinf(outputs)] = 5

    return outputs_incoded
incoded_outputs=band_encoding(model_outputs)

In [10]:
#rescaling dataa

X_train, X_val, y_train, y_val = sklearn.model_selection.train_test_split(
    model_inputs,
    incoded_outputs,
    test_size=0.2,
    random_state=0,
    shuffle=True
)
scaler=sklearn.preprocessing.StandardScaler()
n_feat = X_train.shape[-1]

X_train_2d = X_train.reshape(-1, n_feat)
scaler.fit(X_train_2d)

X_train_scaled = scaler.transform(X_train.reshape(-1, n_feat)).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val.reshape(-1, n_feat)).reshape(X_val.shape)

In [11]:
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=256,
    shuffle=False
)

# model prep and


In [12]:
#definign model
import torch.nn as nn
n_feat=7
classes=6
d_mod=64
n_hed=4
n_lyr=2
dim_feed=128
drp=.1

class PeakTransformer(nn.Module):
    def __init__(
        self,
        n_features=n_feat,
        n_classes=classes,
        d_model=d_mod,
        n_heads=n_hed,
        n_layers=n_lyr,
        dim_feedforward=dim_feed,
        dropout=drp
    ):
        super().__init__()

        self.input_proj = nn.Linear(n_features, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, x):
        """
        x shape: (batch_size, 10, 2)

        returns:
            logits shape: (batch_size, 10, 6)
        """

        x = self.input_proj(x)
        x = self.encoder(x)
        logits = self.classifier(x)

        return logits

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PeakTransformer(
    n_features=n_feat,
    n_classes=classes,
    d_model=d_mod,
    n_heads=n_hed,
    n_layers=n_lyr
).to(device)

counts = torch.bincount(y_train_t.reshape(-1), minlength=6).float()

# inverse-frequency style weights
weights = counts.sum() / (6 * counts)

# avoid problems if one class is missing from training
weights[torch.isinf(weights)] = 0.0
weights = weights.to(device)

print("Class counts:", counts)
print("Class weights:", weights)

Class counts: tensor([1.0052e+04, 2.1171e+04, 4.3907e+04, 4.1848e+04, 1.3000e+01, 1.0981e+05])
Class weights: tensor([3.7604e+00, 1.7855e+00, 8.6091e-01, 9.0327e-01, 2.9077e+03, 3.4423e-01])


# Training DO NOT RUN

In [14]:
#trainging DO NOT RUN

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

n_epochs = 75

for epoch in range(n_epochs):
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)

        # logits shape: (batch, 10, 6)
        # y_batch shape: (batch, 10)
        #
        # flatten so CrossEntropyLoss sees:
        # logits: (batch * 10, 6)
        # labels: (batch * 10)
        loss_class = criterion(
            logits.reshape(-1, 6),
            y_batch.reshape(-1)
        )

        prob = torch.softmax(logits, dim=-1)

        # Sum the probabilities of each sideband class across the 10 peaks
        counts = prob[:, :, :5].sum(dim=1)      # shape = (batch_size, 5)

        # Penalize counts greater than 1
        rule_loss = torch.relu(counts - 1.0).pow(2).mean()
        rule_lamda=20
        loss=loss_class+ (rule_lamda*rule_loss)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)

    train_loss /= len(train_loader.dataset)

    # validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)

            loss_class = criterion(
                        logits.reshape(-1, 6),
                        y_batch.reshape(-1)
                    )
            
            prob = torch.softmax(logits, dim=-1)
    
            # Sum the probabilities of each sideband class across the 10 peaks
            counts = prob[:, :, :5].sum(dim=1)      # shape = (batch_size, 5)
    
            # Penalize counts greater than 1
            rule_loss = torch.relu(counts - 1.0).pow(2).mean()
    
            loss=loss_class+ (rule_lamda*rule_loss)

            val_loss += loss.item() * X_batch.size(0)

            predictions = torch.argmax(logits, dim=-1)

            correct += (predictions == y_batch).sum().item()
            total += y_batch.numel()

    val_loss /= len(val_loader.dataset)
    val_acc = correct / total

    print(
        f"Epoch {epoch + 1:03d} | "
        f"train loss = {train_loss:.4f} | "
        f"val loss = {val_loss:.4f} | "
        f"val acc = {val_acc:.4f}"
    )

Epoch 001 | train loss = 0.8227 | val loss = 0.5187 | val acc = 0.9246
Epoch 002 | train loss = 0.4123 | val loss = 0.2671 | val acc = 0.9560
Epoch 003 | train loss = 0.3098 | val loss = 0.2679 | val acc = 0.9447
Epoch 004 | train loss = 0.2528 | val loss = 0.3175 | val acc = 0.9590
Epoch 005 | train loss = 0.2469 | val loss = 0.3321 | val acc = 0.9345
Epoch 006 | train loss = 0.2409 | val loss = 0.2263 | val acc = 0.9464
Epoch 007 | train loss = 0.2684 | val loss = 0.2288 | val acc = 0.9611
Epoch 008 | train loss = 0.2187 | val loss = 0.2111 | val acc = 0.9573
Epoch 009 | train loss = 0.2099 | val loss = 0.2616 | val acc = 0.9402
Epoch 010 | train loss = 0.2259 | val loss = 0.2350 | val acc = 0.9527
Epoch 011 | train loss = 0.2000 | val loss = 0.1830 | val acc = 0.9572
Epoch 012 | train loss = 0.2039 | val loss = 0.2263 | val acc = 0.9564
Epoch 013 | train loss = 0.1743 | val loss = 0.2110 | val acc = 0.9502
Epoch 014 | train loss = 0.1630 | val loss = 0.2785 | val acc = 0.9632
Epoch 

# saving model

In [17]:
import joblib

checkpoint = {
    "model_state_dict": model.state_dict(),

    # Model architecture
    "n_features": n_feat,
    "n_classes": classes,
    "d_model": d_mod,
    "n_heads": n_hed,
    "n_layers": n_lyr,
    "dim_feedforward": dim_feed,
    "dropout": drp,
}

torch.save(checkpoint, "Trained_models/7_feat_SNR10_SNR50_combo_transformer.pt")
joblib.dump(scaler, "Trained_models/7_feat_SNR10_SNR50_combo_scaler.pkl")

['Trained_models/7_feat_SNR10_SNR50_combo_scaler.pkl']